# DS6 — Step 2 Path A': Behavior-Only Phenotyping (DyS removed)

**Stage 4, Step 2 — Phenotyping, Path A'**
**Dataset:** DS6 (Danish Adolescents, quasi-experimental)
**Date:** 2026-08-06
**Blueprint:** `outputs/reports/stage4/step2_phenotyping/ds6_path_a_behavior_only_blueprint.md`

---

## Trigger and scope

The main Step 2 pass (notebook 140) produced a k=2 split that turned out to be (almost)
exactly DyS's `regime_changed` binary feature (verified: perfect separation). On the radar
chart this meant Profile A showed zero spread on that axis (it IS regime_changed=0 by
construction) -- a real signal that one feature dominated the clustering, not a rendering
bug. This notebook asks the DS5/Structured Path A question directly: **what does the
phenotype structure look like WITHOUT the dominant feature family?**

**Design:** drop DyS entirely (not substitute a different DyS quantity) -- 4 continuous
features: `h_period` (StoP), `hhi_change` (NeT), `H_emp_baseline` (InT), `susceptibility`
(StM). Compare cluster structure to notebook 140 via Adjusted Rand Index (ARI).

---

## 1. Load Feature Sources (4 features, DyS excluded — blueprint Part 1)

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path("__file__").resolve().parent.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from collections import OrderedDict
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score, adjusted_rand_score
from sklearn.decomposition import PCA
from scipy.optimize import linear_sum_assignment
from scipy import stats
import warnings
warnings.filterwarnings("ignore")

MO = PROJECT_ROOT / "outputs" / "model_outputs" / "ds6"
FEATURES = PROJECT_ROOT / "outputs" / "features" / "ds6"
OUT = MO / "phenotyping_path_a"
FIG = OUT / "figures"
FIG.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
print("Output folder:", OUT)

In [ ]:
stop = pd.read_csv(MO / "stochastic_processes" / "participant_stop_parameters.csv")
net_bridge = pd.read_csv(MO / "network_models" / "stop_net_period_comparison.csv")
intt = pd.read_csv(MO / "information_theory" / "participant_entropy_metrics.csv")
stm = pd.read_csv(MO / "statistical_mechanics" / "participant_macrostate_metrics.csv")

# 4 features, no DyS (blueprint Part 1 -- drop the dominant family, don't substitute within it)
FEATURE_MAP = OrderedDict([
    ("h_period",         ("StoP", stop[["participant_id_canonical", "h_period"]])),
    ("hhi_change",       ("NeT",  net_bridge[["participant_id_canonical", "hhi_change"]])),
    ("H_emp_baseline",   ("InT",  intt[["participant_id_canonical", "H_emp_baseline"]])),
    ("susceptibility",   ("StM",  stm[["participant_id_canonical", "susceptibility"]])),
])

feat_df = None
for col, (family, df) in FEATURE_MAP.items():
    feat_df = df if feat_df is None else feat_df.merge(df, on="participant_id_canonical", how="outer")

feature_cols = list(FEATURE_MAP.keys())
complete_case = feat_df.dropna(subset=feature_cols).reset_index(drop=True)

print(f"All participants with any of the 4 features: {len(feat_df)}")
print(f"Complete-case (all 4 features present): {len(complete_case)}")
complete_case.head()

---

## 2. Feature Redundancy Report

In [ ]:
corr_matrix = complete_case[feature_cols].corr()
print(corr_matrix.round(3))

flagged = []
for i, c1 in enumerate(feature_cols):
    for c2 in feature_cols[i+1:]:
        r = corr_matrix.loc[c1, c2]
        if abs(r) > 0.6:
            flagged.append((c1, c2, round(r, 3)))

print()
if flagged:
    print("Pairs with |r| > 0.6:")
    for c1, c2, r in flagged:
        print(f"  {c1} <-> {c2}: r={r}")
else:
    print("No unexpected redundant pairs (|r| > 0.6) among the 4 features.")

corr_matrix.to_csv(OUT / "feature_redundancy_report.csv")

---

## 3. Save Feature Matrix

In [ ]:
complete_case.to_csv(OUT / "phenotype_feature_matrix.csv", index=False)
print(f"Saved: phenotype_feature_matrix.csv ({complete_case.shape})")

---

## 4. Figure — Feature Correlation Matrix

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(corr_matrix, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(feature_cols))); ax.set_xticklabels(feature_cols, rotation=45, ha="right")
ax.set_yticks(range(len(feature_cols))); ax.set_yticklabels(feature_cols)
for i in range(len(feature_cols)):
    for j in range(len(feature_cols)):
        ax.text(j, i, f"{corr_matrix.values[i,j]:.2f}", ha="center", va="center", fontsize=9)
plt.colorbar(im, ax=ax, label="Pearson r")
ax.set_title("DS6 Path A' -- feature correlation matrix")
plt.tight_layout()
plt.savefig(FIG / "fig1_feature_correlation_matrix.png", dpi=150)
plt.show()
print("Saved fig1_feature_correlation_matrix.png")

---

## 5. K-Selection

In [ ]:
X = complete_case[feature_cols].to_numpy()
X_mean, X_std = X.mean(axis=0), X.std(axis=0)
X_z = (X - X_mean) / X_std

N = len(complete_case)
MIN_CELL = max(10, int(0.05 * N))
print(f"N={N}, MIN_CELL={MIN_CELL}")

k_results = {}
for k in range(2, 5):
    km = KMeans(n_clusters=k, n_init=10, random_state=RANDOM_STATE).fit(X_z)
    labels = km.labels_
    sizes = pd.Series(labels).value_counts()
    sil = silhouette_score(X_z, labels)
    ch = calinski_harabasz_score(X_z, labels)
    db = davies_bouldin_score(X_z, labels)
    k_results[k] = {"labels": labels, "silhouette": sil, "CH": ch, "DB": db,
                     "min_size": int(sizes.min()), "sizes": sizes.to_dict()}
    print(f"k={k}: silhouette={sil:.3f}, CH={ch:.1f}, DB={db:.3f}, "
          f"min_cell={sizes.min()}, sizes={dict(sizes)}")

valid_ks = [k for k, r in k_results.items() if r["min_size"] >= MIN_CELL]
print(f"\nValid k (min_cell >= {MIN_CELL}): {valid_ks}")

# NOTE: do NOT silently default to k=2 when valid_ks is empty -- this is exactly the bug
# Structured's own Path D notebook caught ("silent k=2 fallback when no k actually met the
# min_cell floor, producing a spurious STABLE 1-vs-40 split"). Every k here (2,3,4) isolates
# the SAME ~9 participants regardless of k -- verified by inspecting min_size across all
# three: min_size=9 at every single k. That is the signature of a persistent multivariate
# outlier group, not real cluster structure. Report this honestly instead of forcing a k.
NO_FEASIBLE_K = len(valid_ks) == 0
if NO_FEASIBLE_K:
    print(f"\n*** NO FEASIBLE k at MIN_CELL={MIN_CELL} *** — every k tests isolates the same "
          f"~{min(r['min_size'] for r in k_results.values())}-participant group regardless of k. "
          f"This is a genuine outlier-driven signature, not evidence of a real phenotype split. "
          f"Proceeding with k=2 ONLY as an exploratory/provisional characterization of that "
          f"outlier group (labeled as such throughout below), not as a validated cluster solution.")
SELECTED_K = max(valid_ks, key=lambda k: k_results[k]["silhouette"]) if valid_ks else 2
print(f"Selected k: {SELECTED_K}" + (" (PROVISIONAL -- outlier-driven, not min-cell-valid)" if NO_FEASIBLE_K else ""))

---

## 6. Figure — K-Selection Metrics

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
ks = list(k_results.keys())
axes[0].plot(ks, [k_results[k]["silhouette"] for k in ks], "o-"); axes[0].set_title("Silhouette"); axes[0].set_xlabel("k")
axes[1].plot(ks, [k_results[k]["CH"] for k in ks], "o-", color="orange"); axes[1].set_title("Calinski-Harabasz"); axes[1].set_xlabel("k")
axes[2].plot(ks, [k_results[k]["DB"] for k in ks], "o-", color="green"); axes[2].set_title("Davies-Bouldin (lower better)"); axes[2].set_xlabel("k")
for ax in axes:
    ax.axvline(SELECTED_K, color="red", linestyle="--", alpha=0.5)
plt.suptitle(f"DS6 Path A' -- k-selection metrics (selected k={SELECTED_K})")
plt.tight_layout()
plt.savefig(FIG / "fig2_k_selection_metrics.png", dpi=150)
plt.show()
print("Saved fig2_k_selection_metrics.png")

---

## 7. Bootstrap Jaccard Stability Check

In [ ]:
def bootstrap_jaccard(X_z, labels_full, k, n_boot=100, frac=0.8, seed=RANDOM_STATE):
    n = len(X_z)
    rng = np.random.RandomState(seed)
    jaccards = []
    for b in range(n_boot):
        idx = rng.choice(n, size=int(n * frac), replace=False)
        km_sub = KMeans(n_clusters=k, n_init=10, random_state=seed + b).fit(X_z[idx])
        labels_sub = km_sub.labels_
        labels_full_sub = labels_full[idx]

        cont = np.zeros((k, k))
        for i in range(k):
            for j in range(k):
                cont[i, j] = np.sum((labels_full_sub == i) & (labels_sub == j))
        row_ind, col_ind = linear_sum_assignment(-cont)

        pair_jaccards = []
        for i, j in zip(row_ind, col_ind):
            set_full = set(np.where(labels_full_sub == i)[0])
            set_sub = set(np.where(labels_sub == j)[0])
            union = len(set_full | set_sub)
            pair_jaccards.append(len(set_full & set_sub) / union if union > 0 else np.nan)
        jaccards.append(np.nanmean(pair_jaccards))
    return float(np.mean(jaccards)), float(np.std(jaccards))

jac_mean, jac_std = bootstrap_jaccard(X_z, k_results[SELECTED_K]["labels"], SELECTED_K)
stability_verdict = "STABLE" if jac_mean >= 0.75 else "MARGINAL" if jac_mean >= 0.60 else "UNSTABLE"
if NO_FEASIBLE_K:
    stability_verdict += " (BUT PROVISIONAL -- this k was never min-cell-valid; a high Jaccard here just means the 9-person outlier group is consistently re-identified across resamples, which is expected for extreme outliers and does NOT indicate genuine 2-phenotype structure)"
print(f"Bootstrap Jaccard (k={SELECTED_K}, 100 resamples, 80% subsample): "
      f"mean={jac_mean:.3f}, std={jac_std:.3f}")
print(f"Stability verdict: {stability_verdict}")

---

## 8. PCA Sanity Check

In [ ]:
pca = PCA(n_components=len(feature_cols))
pca.fit(X_z)
print("Variance explained per component:")
for i, v in enumerate(pca.explained_variance_ratio_):
    print(f"  PC{i+1}: {v*100:.1f}%")
print(f"Cumulative PC1+PC2: {pca.explained_variance_ratio_[:2].sum()*100:.1f}%")

fig, ax = plt.subplots(figsize=(7, 6))
coords = pca.transform(X_z)[:, :2]
labels_sel = k_results[SELECTED_K]["labels"]
for lab in np.unique(labels_sel):
    mask = labels_sel == lab
    ax.scatter(coords[mask, 0], coords[mask, 1], label=f"Cluster {lab} (n={mask.sum()})", alpha=0.7)
ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)")
ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)")
ax.set_title(f"DS6 Path A' -- PCA biplot, k={SELECTED_K}")
ax.legend()
plt.tight_layout()
plt.savefig(FIG / "fig3_pca_biplot.png", dpi=150)
plt.show()
print("Saved fig3_pca_biplot.png")

---

## 9. Cluster Assignments and Profiles

In [ ]:
PROFILE_NAMES = [f"Profile {chr(65+i)}'" for i in range(SELECTED_K)]
label_to_name = {lab: PROFILE_NAMES[i] for i, lab in enumerate(sorted(np.unique(labels_sel)))}

complete_case["cluster_label"] = [label_to_name[l] for l in labels_sel]

cluster_assignments = complete_case[["participant_id_canonical", "cluster_label"] + feature_cols].copy()
cluster_assignments.to_csv(OUT / "cluster_assignments.csv", index=False)

cluster_profiles = complete_case.groupby("cluster_label")[feature_cols].agg(["mean", "median", "std"])
cluster_profiles.to_csv(OUT / "cluster_profiles.csv")

print(f"Saved: cluster_assignments.csv ({cluster_assignments.shape}), cluster_profiles.csv")
print()
print("Cluster sizes:")
print(complete_case["cluster_label"].value_counts())
print()
print("Cluster means (raw, unstandardized):")
print(complete_case.groupby("cluster_label")[feature_cols].mean().round(4))

---

## ⚠ Important caveat before reading Section 10 onward

**No k met the min_cell floor (MIN_CELL=11)** — k=2, k=3, and k=4 all isolate the SAME
~9-participant group regardless of how many clusters are requested (min_size=9 at every k
tested). Inspecting those 9 participants directly shows why: they are genuine multivariate
outliers with susceptibility 4-10x the population mean (0.26-0.58 vs population mean 0.046)
combined with unusually high hhi_change. This is an **outlier-detection signature, not
evidence of a real second phenotype** in the remaining 218 participants.

**What this notebook does from here:** proceeds with k=2 anyway, but every downstream
result (cluster profiles, ARI, radar chart, overlay) describes "218 typical participants vs.
9 extreme-susceptibility outliers," not two balanced behavioral phenotypes. Treat
"Profile B'" (n=9) as an outlier-characterization exercise, not a validated cluster. This is
the same failure mode Structured's own Path D notebook caught ("every candidate k was
driven by one extreme outlier, reported honestly as infeasible rather than forced") —
same discipline applied here.

---

## 10. ARI Comparison to the Main Pass (notebook 140) — blueprint Part 2

Read with the caveat above in mind: an ARI against a 218-vs-9 outlier split answers a
narrower question than originally intended ("does the main pass's k=2 solution align with
an outlier group in the 4 continuous features?") rather than the cleaner "does phenotype
structure reproduce without DyS?" question the blueprint posed. Still informative — see
Section 10 interpretation below — but not the clean comparison originally planned.

In [ ]:
main_clusters = pd.read_csv(MO / "phenotyping" / "cluster_assignments.csv")
merged_ari = complete_case[["participant_id_canonical", "cluster_label"]].merge(
    main_clusters[["participant_id_canonical", "cluster_label"]],
    on="participant_id_canonical", how="inner", suffixes=("_path_a", "_main")
)
print(f"Overlapping participants (Path A' AND main pass): {len(merged_ari)}")

ari = adjusted_rand_score(merged_ari["cluster_label_main"], merged_ari["cluster_label_path_a"])
print(f"Adjusted Rand Index (Path A' vs main pass, k={SELECTED_K} vs k=2): {ari:.4f}")

if ari > 0.5:
    verdict = "HIGH -- phenotype structure reproduces without DyS; the main pass's split was not specific to regime_changed"
elif ari > 0.15:
    verdict = "MODERATE -- partial overlap; DyS contributes real but not exclusive structure"
else:
    verdict = "LOW/ORTHOGONAL -- removing DyS changes the split substantially; the main pass's k=2 solution was substantially DyS's own signal"
print(f"Verdict: {verdict}")

# Cross-tab for direct inspection
print()
print("Cross-tab (main pass label x Path A' label):")
print(pd.crosstab(merged_ari["cluster_label_main"], merged_ari["cluster_label_path_a"]))

ari_comparison = pd.DataFrame([{
    "n_overlap": len(merged_ari), "k_main": 2, "k_path_a": SELECTED_K,
    "ARI": round(ari, 4), "verdict": verdict,
}])
ari_comparison.to_csv(OUT / "ari_comparison.csv", index=False)
print("\nSaved ari_comparison.csv")

---

## 11. The Enhanced Radar Chart — 4 continuous axes, no degenerate binary axis

In [ ]:
FONT_SIZES = {
    "axis_labels": 15, "section_layers": 13, "grid_ticks": 12,
    "bottom_legends": 13, "legend_titles": 14,
}

RADAR_DIMS = OrderedDict([
    ("Trend\nshift",             "h_period"),           # StoP
    ("Predictability\n(baseline)", "H_emp_baseline"),   # InT
    ("Network\nconcentration shift", "hhi_change"),     # NeT
    ("Configuration\nsusceptibility", "susceptibility"), # StM
])

AXIS_MODEL = {
    "Trend\nshift": "StoP", "Predictability\n(baseline)": "InT",
    "Network\nconcentration shift": "NeT", "Configuration\nsusceptibility": "StM",
}

MODEL_COLORS = {
    "NeT": "#4CAF50", "InT": "#2196F3", "StoP": "#FF9800", "StM": "#9C27B0",
}

PROFILE_COLOR_CYCLE = ["#A20A7C", "#7E57C2", "#00897B", "#EF6C00"]
PROFILE_COLORS = {name: PROFILE_COLOR_CYCLE[i] for i, name in enumerate(PROFILE_NAMES)}

# 2-2 grouping over the RADAR_DIMS order above (no DyS/regime section this time)
SECTION_LAYERS = [
    {"name": "Volume & Predictability", "dim_idx": [0, 1], "color": "#C62828"},
    {"name": "Structure Change",        "dim_idx": [2, 3], "color": "#0A4B96"},
]

print("Radar configuration set for", len(RADAR_DIMS), "axes,", SELECTED_K, "profiles.")

In [ ]:
def normalize_features(df, dim_feats):
    norm = {}
    for feat in dim_feats:
        v = df[feat].dropna()
        norm[feat] = (df[feat] - v.min()) / (v.max() - v.min() + 1e-12)
    return norm


def draw_model_arcs(ax, angles, dim_labels, axis_model, model_colors, arc_r=1.04, arc_lw=3.5, alpha=0.45):
    n = len(angles)
    half_gap = (2 * np.pi / n) * 0.40
    for label, angle in zip(dim_labels, angles):
        model = axis_model.get(label, "")
        color = model_colors.get(model, "#AAAAAA")
        theta = np.linspace(angle - half_gap, angle + half_gap, 40)
        ax.plot(theta, np.full_like(theta, arc_r), color=color, lw=arc_lw,
                solid_capstyle="round", alpha=alpha, clip_on=False, zorder=10)


def draw_section_layers(ax, angles, section_layers, band_inner=1.09, band_outer=1.18, label_r=1.135):
    step = angles[1] - angles[0]
    half = step / 2
    for sec in section_layers:
        idx = sec["dim_idx"]
        color = sec["color"]
        a0 = angles[idx[0]] - half
        a1 = angles[idx[-1]] + half
        theta = np.linspace(a0, a1, 300)
        ax.fill_between(theta, band_inner, band_outer, color=color, alpha=0.13, clip_on=False, zorder=5)
        ax.plot(theta, np.full_like(theta, band_inner), color=color, lw=0.8, alpha=0.40, clip_on=False, zorder=6)
        ax.plot(theta, np.full_like(theta, band_outer), color=color, lw=0.8, alpha=0.40, clip_on=False, zorder=6)


def draw_axis_label_boxes(ax, angles, dim_labels, label_r=1.30):
    for label, angle in zip(dim_labels, angles):
        ax.text(angle, label_r, label, ha="center", va="center",
                fontsize=FONT_SIZES["axis_labels"], color="#222222",
                clip_on=False, zorder=25, multialignment="center",
                bbox=dict(boxstyle="round,pad=0.30", fc="white", ec="#BBBBBB", lw=0.9, alpha=0.97))


def compute_iqr_band(sub_df, norm_cols, angles):
    q25 = sub_df[norm_cols].quantile(0.25).to_numpy(dtype=float)
    q75 = sub_df[norm_cols].quantile(0.75).to_numpy(dtype=float)
    mean = sub_df[norm_cols].mean(skipna=True).to_numpy(dtype=float)
    def close(arr): return np.append(arr, arr[0])
    return close(q25), close(q75), close(mean)


def plot_single_profile(ax, sub_df, norm_cols, angles, profile_name, color, dim_labels,
                         axis_model, model_colors, section_layers,
                         individual_alpha=0.25, individual_lw=0.7, mean_lw=2.8):
    for _, row in sub_df.iterrows():
        v = np.append(row[norm_cols].to_numpy(dtype=float), row[norm_cols].iloc[0])
        ax.plot(angles, v, color=color, alpha=individual_alpha, lw=individual_lw, zorder=1)

    q25, q75, mean_vals = compute_iqr_band(sub_df, norm_cols, angles)
    ax.fill_between(angles, q25, q75, color=color, alpha=0.22, zorder=2)
    ax.plot(angles, q25, color=color, alpha=0.45, lw=1.0, ls="--", zorder=4)
    ax.plot(angles, q75, color=color, alpha=0.45, lw=1.0, ls="--", zorder=4)

    ax.plot(angles, mean_vals, "o-", color=color, lw=mean_lw, ms=5,
            markerfacecolor="white", markeredgecolor=color, markeredgewidth=1.8,
            label=f"{profile_name} mean  (n={len(sub_df)})", zorder=5)
    ax.fill(angles, mean_vals, color=color, alpha=0.10, zorder=3)

    draw_model_arcs(ax, angles[:-1], dim_labels, axis_model, model_colors)
    draw_section_layers(ax, angles[:-1], section_layers)
    draw_axis_label_boxes(ax, angles[:-1], dim_labels)

    ax.set_xticks(angles[:-1])
    ax.set_xticklabels([""] * len(dim_labels))
    ax.tick_params(axis="x", length=0)
    ax.set_ylim(0, 1)
    ax.set_yticks([0.25, 0.50, 0.75, 1.00])
    ax.set_yticklabels(["0.25", "0.50", "0.75", "1.00"], fontsize=FONT_SIZES["grid_ticks"],
                        alpha=0.5, color="#555555")
    ax.spines["polar"].set_color("#272727")
    ax.grid(color="#121111", lw=0.6)


def plot_phenotype_profiles_enhanced(pop_df, profile_col, participant_col, profile_names,
                                       profile_colors, save_path=None):
    available = OrderedDict((k, v) for k, v in RADAR_DIMS.items() if v in pop_df.columns)
    dim_labels = list(available.keys())
    dim_feats = list(available.values())
    am = {k: AXIS_MODEL[k] for k in dim_labels}

    norm_map = normalize_features(pop_df, dim_feats)
    radar_df = pop_df.copy()
    for feat in dim_feats:
        radar_df[f"{feat}_norm"] = norm_map[feat]
    norm_cols = [f"{feat}_norm" for feat in dim_feats]

    angles = np.linspace(0, 2 * np.pi, len(dim_labels), endpoint=False)
    angles = np.append(angles, angles[0])

    n_profiles = len(profile_names)
    fig, axes = plt.subplots(1, n_profiles, figsize=(9 * n_profiles, 11.5), subplot_kw=dict(polar=True))
    if n_profiles == 1:
        axes = [axes]
    fig.subplots_adjust(wspace=0.7, left=0.06, right=0.94, top=0.82, bottom=0.27)

    for pname, ax in zip(profile_names, axes):
        color = profile_colors[pname]
        sub = radar_df[radar_df[profile_col] == pname].copy()
        if sub.empty:
            ax.set_title(f"{pname}\n(no data)")
            continue
        plot_single_profile(ax=ax, sub_df=sub, norm_cols=norm_cols, angles=angles,
                             profile_name=pname, color=color, dim_labels=dim_labels,
                             axis_model=am, model_colors=MODEL_COLORS, section_layers=SECTION_LAYERS)

    def _desc(m):
        return {"InT": "Information Theory", "StoP": "Stochastic Processes", "StM": "Statistical Mechanics",
                "NeT": "Network Theory"}.get(m, m)

    graph_patches = [mpatches.Patch(facecolor=profile_colors[p], label=f"{p} mean") for p in profile_names]
    graph_patches.append(mpatches.Patch(facecolor="grey", alpha=0.35, label="IQR band (25th-75th pct.)"))

    arc_patches = [mpatches.Patch(color=c, label=f"{m} \u2014 {_desc(m)}") for m, c in MODEL_COLORS.items()]
    sec_patches = [mpatches.Patch(color=s["color"], alpha=0.55, label=s["name"]) for s in SECTION_LAYERS]

    leg0 = fig.legend(handles=graph_patches, loc="upper center", bbox_to_anchor=(0.5, 1.02), ncol=1,
                       fontsize=FONT_SIZES["bottom_legends"], framealpha=0.95, edgecolor="#CCCCCC",
                       title="Profiles  (mean line + IQR)", title_fontsize=FONT_SIZES["legend_titles"])
    fig.add_artist(leg0)
    leg1 = fig.legend(handles=arc_patches, loc="lower center", bbox_to_anchor=(0.28, 0.00), ncol=2,
                       columnspacing=1.0, handletextpad=0.6,
                       fontsize=FONT_SIZES["bottom_legends"], framealpha=0.95, edgecolor="#CCCCCC",
                       title="Model families  (axis color)", title_fontsize=FONT_SIZES["legend_titles"])
    fig.add_artist(leg1)
    fig.legend(handles=sec_patches, loc="lower center", bbox_to_anchor=(0.76, 0.00), ncol=1,
               fontsize=FONT_SIZES["bottom_legends"], framealpha=0.95, edgecolor="#CCCCCC",
               title="Conceptual layers  (band color)", title_fontsize=FONT_SIZES["legend_titles"])

    if save_path is not None:
        fig.savefig(save_path, bbox_inches="tight", dpi=300)
        print(f"Saved: {save_path}")
    plt.show()
    return fig, axes

fig, axes = plot_phenotype_profiles_enhanced(
    pop_df=complete_case,
    profile_col="cluster_label",
    participant_col="participant_id_canonical",
    profile_names=PROFILE_NAMES,
    profile_colors=PROFILE_COLORS,
    save_path=FIG / "fig4_phenotype_profiles_enhanced.png",
)

---

## 12. Psychosocial Overlay

In [ ]:
syd = pd.read_csv(MO / "system_dynamics" / "participant_stock_flow.csv")

overlay_df = complete_case.merge(
    syd[["participant_id_canonical", "addiction_index_change", "self_control_index_change"]],
    on="participant_id_canonical", how="inner"
)
print(f"N with cluster assignment AND SyD prepost data: {len(overlay_df)}")

SCALES = ["addiction_index_change", "self_control_index_change"]

overlay_results = []
if len(overlay_df) >= 10 and overlay_df["cluster_label"].nunique() >= 2:
    for scale in SCALES:
        groups = [g[scale].dropna().values for _, g in overlay_df.groupby("cluster_label") if len(g[scale].dropna()) >= 3]
        if len(groups) < 2:
            continue
        h_stat, p_val = stats.kruskal(*groups)
        overlay_results.append({"scale": scale, "H_stat": round(h_stat, 3), "p_raw": round(p_val, 4)})
    overlay_df_result = pd.DataFrame(overlay_results)
    if len(overlay_df_result) > 0:
        overlay_df_result["p_bonferroni"] = (overlay_df_result["p_raw"] * len(overlay_df_result)).clip(upper=1.0)
        print(overlay_df_result.to_string(index=False))
    else:
        print("Not enough non-null data per scale/cluster for a Kruskal-Wallis test.")
        overlay_df_result = pd.DataFrame(columns=["scale", "H_stat", "p_raw", "p_bonferroni"])
else:
    print(f"N too small ({len(overlay_df)}) or single-cluster -- skipping formal test.")
    overlay_df_result = pd.DataFrame(columns=["scale", "H_stat", "p_raw", "p_bonferroni"])

overlay_df_result.to_csv(OUT / "psychosocial_overlays.csv", index=False)
print(f"\nSaved: psychosocial_overlays.csv")

---

## 13. model_run_summary.md

In [ ]:
from datetime import datetime
MODEL_RUN_TS = datetime.now().strftime('%Y-%m-%d %H:%M:%S')

no_k_note = (
    "\n## \u26a0 NO FEASIBLE k -- READ THIS FIRST\n\n"
    "k=2/3/4 all isolate the SAME ~9-participant group regardless of k (min_size=9 at every "
    "k tested) -- a multivariate-outlier signature (susceptibility 4-10x the population mean "
    "combined with elevated hhi_change), NOT evidence of a real second phenotype. The k=2 "
    "result below is reported as an outlier characterization only, not a validated cluster "
    "solution. Same failure mode Structured's own Path D notebook already caught once.\n"
    if NO_FEASIBLE_K else ""
)

summary = f"""# DS6 Path A' (Behavior-Only, DyS removed) -- Model Run Summary

**Run at:** {MODEL_RUN_TS}
**Notebook:** notebooks/141_ds6_behavior_only_phenotype.ipynb
**Blueprint:** outputs/reports/stage4/step2_phenotyping/ds6_path_a_behavior_only_blueprint.md
{no_k_note}
## Trigger
User inspected the main pass's radar chart and asked why Profile A showed nothing on the
regime-change axis. Root cause verified: that axis perfectly separated the k=2 clusters
(Profile A = regime_changed=0 for all 125), so it had zero within-cluster variance for
Profile A by construction -- a real dominance signal, not a rendering bug. This notebook
tests the phenotype structure with that dominant feature removed entirely.

## Sample
- Complete-case population (4 behavioral features, no DyS): N={len(complete_case)}

## Feature redundancy
{"No unexpected redundant pairs (|r|>0.6)." if not flagged else str(flagged)}

## K-selection
"""
for k in k_results:
    r = k_results[k]
    summary += f"- k={k}: silhouette={r['silhouette']:.3f}, CH={r['CH']:.1f}, DB={r['DB']:.3f}, min_cell={r['min_size']}, sizes={r['sizes']}\n"
summary += f"""
**Selected k={SELECTED_K}** (silhouette={k_results[SELECTED_K]['silhouette']:.3f}, subject to min_cell>={MIN_CELL})

## Stability
- Bootstrap Jaccard (100 resamples, 80% subsample): mean={jac_mean:.3f}, std={jac_std:.3f}
- Verdict: {stability_verdict}

## PCA
- PC1+PC2 variance explained: {pca.explained_variance_ratio_[:2].sum()*100:.1f}%

## Cluster profiles (raw means)
{complete_case.groupby("cluster_label")[feature_cols].mean().round(4).to_string()}

## Cluster sizes
{complete_case["cluster_label"].value_counts().to_string()}

## ARI comparison to the main pass (notebook 140)
- Overlap N: {len(merged_ari)}
- ARI: {ari:.4f}
- Verdict: {verdict}

## Psychosocial overlays (N={len(overlay_df)})
{overlay_df_result.to_string(index=False) if len(overlay_df_result) > 0 else "Inconclusive -- N too small or single cluster."}

## Key limitations
1. Removing DyS entirely (not substituting) is a deliberate test of dominance, not a claim
   that DyS's information is worthless -- see blueprint Part 1
2. 4 features only -- narrower feature space than the main pass's 5
3. Cluster labels here (Profile A'/B'/...) are a SEPARATE KMeans fit, not a subset of the
   main pass's labels -- compare via ARI (Section 10), not by assuming label correspondence
"""

with open(OUT / "model_run_summary.md", "w") as f:
    f.write(summary)

print(summary)